# Two-tone coherence

Elhilali et al. (2009) sweep the onset asynchrony of two tones from synchrony to alternation and
plot how segregated their model thinks the pair is. Figure 8B. Nobody has measured that curve in
a listener; their own psychophysics (Fig. 2) compares two states, and through a tempo difference
rather than a fixed lag.

This is that experiment. Their asynchrony-detection task, with ΔT as a continuous variable.

In [ ]:
#@title setup
import sys, subprocess, importlib.util, json, math
from pathlib import Path
REF='e40ba027b457efd3482e4c404ea299600f72ed9f'
REPO='https://github.com/MeysamAmirsardari/SeqSFG_task.git'
try:
    import tcoh
except Exception:
    if bool(importlib.util.find_spec('google') and importlib.util.find_spec('google.colab')):
        ROOT=Path('/content')/('tcoh-'+REF[:12])
        if not ROOT.exists():
            subprocess.run(['git','clone','--no-checkout',REPO,str(ROOT)],check=True)
            subprocess.run(['git','-C',str(ROOT),'checkout','--detach',REF],check=True)
    else:
        ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'tcoh/model.py').exists()),None)
    for _m in [k for k in list(sys.modules) if k=='tcoh' or k.startswith('tcoh.')]: del sys.modules[_m]
    sys.path.insert(0,str(ROOT))

get_ipython().run_line_magic('matplotlib','inline')
import numpy as np, matplotlib
matplotlib.rcParams['figure.dpi']=120
import matplotlib.pyplot as plt
from IPython.display import Audio, display, Markdown
from tcoh.config import DEFAULT, validate, conditions
from tcoh import model as M, stimulus as S, plots as P, verify as V
CFG=DEFAULT; D=validate(CFG); CONDS=conditions(CFG)
def head(s): display(Markdown(s))
head(f"A {CFG.f_a_hz:.0f} Hz / B {D.f_b_hz:.0f} Hz, {CFG.df_semitones:g} st apart "
     f"({D.erbs_apart:.1f} ERB) &nbsp; · &nbsp; {CFG.tone_ms:g} ms tones, {CFG.soa_ms:g} ms SOA, "
     f"{CFG.n_tones} per channel &nbsp; · &nbsp; `{CFG.hash()}`")

---
## One trial

Two sounds. In one of them the last high tone is out of place. Which one?

In [ ]:
#@title the trial
P.trial(CFG); plt.show()

---
## Listen

Shift set to 25 ms so it's audible. First interval is always the odd one out here.
Going down the list the task changes character: at 0% you hear a chord split, by 100% there's
no chord left and you're judging a rhythm.

In [ ]:
#@title one clip per asynchrony
for c in [x for x in CONDS if x.a_kind=='coherent']:
    tr=S.build_trial(CFG,c,25.0,np.random.default_rng(4),target_position=1,direction=+1)
    head(f"**ΔT = {c.lag_pct:g}%** &nbsp; lag {CFG.lag_ms(c.lag_pct):.1f} ms &nbsp; · &nbsp; "
         f"model λ₂/λ₁ = {D.model_by_condition[c.name]:.3f}")
    display(Audio(S.render_trial(CFG,tr,D), rate=CFG.sample_rate, normalize=False))

In [ ]:
#@title the two ends of the scale
for name,note in (('b_only','low tone off entirely — the ceiling'),
                  ('scr_0','low tone there and synchronous at the end, but its earlier tones scrambled')):
    c=next(x for x in CONDS if x.name==name)
    tr=S.build_trial(CFG,c,25.0,np.random.default_rng(4),target_position=1,direction=+1)
    head(f"**{name}** — {note}")
    display(Audio(S.render_trial(CFG,tr,D), rate=CFG.sample_rate, normalize=False))

---
## The sweep

Only the outlined tone moves. The low tone never moves. The high tone's grid is the same in
every row, so whatever cue it offers on its own can't vary with ΔT.

In [ ]:
#@title all five conditions
P.schematic(CFG); plt.show()

---
## The model

Re-implemented here rather than read off their figure, because we need the prediction for these
stimuli. One snag: the published equation as written doesn't reproduce the published numbers.
Alternating channels come out anti-correlated, not uncorrelated. Rectifying the product first —
which is what "coincidence detection" means, and what their text describes — gives both values
they quote.

In [ ]:
#@title check against Figure 8
r=M.reproduce_figure8()
rows=["| reading | ΔT=100% | ΔT=0% | monotone |","|---|---|---|---|"]
for k,v in r['readings'].items():
    rows.append(f"| {'**'+k+'**' if k==r['best'] else k} | {v['alternating']:.3f} | "
                f"{v['synchronous']:.3f} | {v['monotone']} |")
rows.append(f"| _published_ | _{M.PUBLISHED['alternating']}_ | _{M.PUBLISHED['synchronous']}_ | _yes_ |")
head("\n".join(rows))

It also settled the tone duration. ΔT is only an ordered axis when the tone fills half the
period. Put a gap in and the predicted index peaks near 75% and comes back down, which would
make a monotone behavioural result meaningless. The validator now refuses other duty cycles.

In [ ]:
#@title duty cycle
scan=M.duty_cycle_scan(soa_ms=CFG.soa_ms, n_tones=CFG.n_tones)
rows=["| tone / SOA | monotone in ΔT? | peak |","|---|---|---|"]
for duty,v in scan.items():
    mark='**'+format(duty,'.3f')+'**' if abs(duty-CFG.duty)<1e-9 else format(duty,'.3f')
    rows.append(f"| {mark} | {'yes' if v['monotone'] else '**no**'} | ΔT = {v['argmax_pct']:.0f}% |")
head("\n".join(rows))

In [ ]:
#@title the prediction
P.prediction(CFG); plt.show()
head("Band = eight defensible readings of the filter bank. The ordering holds across all of "
     "them; the heights don't, so the shape isn't treated as evidence.")

---
## Why the control decides it

Three accounts all predict thresholds rising with ΔT: coherence, interval discrimination, and
plain acoustic overlap. So a rising curve settles nothing — in simulation the Weber rival
produces one on 100% of sessions.

The control holds the final A–B interval exactly and removes only the low tone's *sequence*.
Both rivals depend on nothing but that final pair, so both predict a flat zero. Coherence
predicts a big difference at synchrony that disappears at alternation.

In [ ]:
#@title predicted interaction
coh={c.lag_pct:D.model_by_condition[c.name] for c in CONDS if c.a_kind=='coherent'}
ctl={c.lag_pct:D.model_by_condition[c.name] for c in CONDS if c.a_kind=='scrambled'}
rows=["| ΔT | coherent | scrambled | difference |","|---|---|---|---|"]
for p in sorted(set(coh)&set(ctl)):
    rows.append(f"| {p:g}% | {coh[p]:.3f} | {ctl[p]:.3f} | **{ctl[p]-coh[p]:+.2f}** |")
head("\n".join(rows))
head("A pedestal or overlap account predicts a column of zeros there.")

---
## Checks

Everything that can be checked without a listener.

In [ ]:
#@title run the battery
for section,checks in V.run_battery(CFG, quick=True).items():
    head(f"**{section}**")
    for c in checks:
        head(f"- {'✅' if c.passed else '❌'} {c.name}  \n  <small>{c.detail}</small>")

---
## Power

100 simulated sessions per generating truth. Second row is the point.

| truth | H1 fires | H2 fires |
|---|---|---|
| coherence | 100% | **87%** |
| Weber rival | **100%** | 4% |
| null | 6% | 3% |

Cutting the control from five ΔT levels to three keeps H1 at 99% and drops H2 to 9%. That
version is a screen, not a test.

---
## A simulated run

Not a listener. This is an observer generated from the hypothesis, here to show what comes out
and that the analysis can tell the accounts apart.

In [ ]:
#@title simulate and analyse
import tempfile
from tcoh.runner import Runner
from tcoh.analysis import analyse, coherence_index, interaction_test, load, thresholds
r=Runner(CFG, Path(tempfile.mkdtemp()), audio=False, auto='coherence', seed=11)
sdir=r.run(code='SIM')
res=thresholds(load([sdir])[0], CFG)
idx=coherence_index(res,CFG,n_boot=1200)
ctl=coherence_index(res,CFG,n_boot=1200,a_kind='scrambled')
inter=interaction_test(res,CFG,n_boot=1200)
P.curve(CFG,idx,ctl,simulated=True); plt.show()
P.controls(CFG,inter,simulated=True); plt.show()

In [ ]:
#@title same thing under the rival
r2=Runner(CFG, Path(tempfile.mkdtemp()), audio=False, auto='pedestal', seed=11)
i2=interaction_test(thresholds(load([r2.run(code='SIM')])[0], CFG), CFG, n_boot=1200)
rows=["| generating truth | slope | p | verdict |","|---|---|---|---|"]
for nm,t in (('coherence',inter),('pedestal rival',i2)):
    rows.append(f"| {nm} | {t['slope_per_pct']*100:+.2f} | {t['p_slope_negative']:.4f} | "
                f"{'**coherence**' if t['p_slope_negative']<0.05 else 'not resolved'} |")
head("\n".join(rows))

In [ ]:
#@title full report
print(analyse([sdir], n_boot=1200))

---
## Limits

- The shape test is close to worthless. Over these ΔT levels the model curve and a straight line
  correlate at 0.988.
- Onset lag and overlap are perfectly confounded at 50% duty. Separating them needs a duty-cycle
  experiment; the config supports it.
- The two controls fail in different directions — scrambled matches tone count but isn't fully
  decoherent, pair-only has no A sequence but five fewer tones. The argument needs both to agree.
- κ is normalised by two conditions from the same session. A bad floor or ceiling moves the whole
  curve, so both are printed in ms next to it.
- One listener is one listener.

Hypotheses, decision rules and exclusions are in `tcoh/PREREGISTRATION.md`.

Session is about two hours, meant to be split. `python -m tcoh run --data data --code P01`, then
`--resume` next time.